## [1] Transformer와 비교해 변경이 필요한 부분

원본 챗봇 노트북의 `Transformer` 클래스(Encoder-Decoder 구조, `MultiHeadAttention`,
`PoswiseFeedForwardNet`, `EncoderLayer`, `DecoderLayer`, `Encoder`, `Decoder`)를
기반으로, GPT-1(Decoder-only) 구조로 만들기 위해 아래와 같이 변경

1) **Encoder 클래스 및 enc_emb(인코더 임베딩) 완전 제거**
   - 원본: `enc_emb` → `Encoder` → `enc_out` 을 만들어 Decoder에 전달
   - 변경: Encoder 관련 코드 전부 삭제. Question과 Answer를 하나의 시퀀스로
     합쳐 Decoder(만 있는 구조)에 직접 입력.

2) **DecoderLayer 내 `enc_dec_attn` (Encoder-Decoder Attention) 서브레이어 제거**
   - 원본 `DecoderLayer.forward`: self_attn → norm → **enc_dec_attn** → norm → ffn → norm
   - 변경: self_attn → norm → ffn → norm (2개 서브레이어만 유지)
   - `enc_out`, `dec_enc_mask` 인자도 forward에서 제거

3) **Positional Encoding 방식 변경 (sinusoidal → learned)**
   - 원본: `positional_encoding()` 함수로 만든 고정 sinusoidal 값을 buffer로 등록
   - 변경: GPT-1 논문 수식 h0 = U·We + Wp 그대로, **학습되는 `nn.Embedding`
     기반 position embedding(Wp)**을 사용

4) **마스크 구성 단순화**
   - 원본: `generate_masks()`가 enc_mask, dec_enc_mask, dec_mask(self-attn용) 3종류 생성
   - 변경: 전체 시퀀스에 대한 **causal self-attention mask 1종류만** 생성
     (padding mask + lookahead mask 결합, 원본의 `generate_padding_mask`,
     `generate_lookahead_mask` 함수는 그대로 재사용)

5) **활성함수 변경 (ReLU → GELU)**
   - 원본 `PoswiseFeedForwardNet`: `nn.ReLU()`
   - 변경: GPT-1 논문 4.1절 명시사항에 따라 `nn.GELU()`로 교체

6) **학습 목적함수 변경 (Seq2Seq loss → Causal LM loss)**
   - 원본 `train_step`: `src`(질문)를 encoder에, `tgt_in`(답변 shift)을 decoder에 넣어
     번역 loss 계산
   - 변경: 질문+답변을 합친 **하나의 시퀀스**를 한 칸 shift하여 다음 토큰 예측
     loss만 계산 (`loss_function`, `LearningRateScheduler`는 그대로 재사용)

`MultiHeadAttention`은 self-attention 계산 로직 자체는 동일하므로 **변경 없이 그대로 재사용**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import os
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


## [평가기준 2] 모델의 입력 형태에 맞게 전처리 수행

### Step 1. 데이터 다운로드

`songys/Chatbot_data`의 `ChatbotData.csv` (질문 Q, 답변 A 약 11,823쌍)를 사용합니다.

In [4]:
df = pd.read_csv("./ChatbotData.csv")
print(df.shape)
df.head()

(11823, 3)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


### Step 2. 데이터 정제

원본 노트북의 `preprocess_sentence()`를 챗봇 Step 2 요구사항(영문 소문자화,
한글/영문/숫자/주요 특수문자 외 제거)에 맞게 확장하여 재사용합니다.

In [5]:
# 원본 preprocess_sentence()를 한글 처리가 가능하도록 정규식만 확장 (재사용 + 수정)
def preprocess_sentence(sentence):
    sentence = str(sentence).lower()                              # 대문자 -> 소문자 (원본 유지)
    sentence = re.sub(r"[^0-9a-zA-Zㄱ-힣?.!,¿]+", " ", sentence)   # 한글/영문/숫자/주요 특수문자 외 제거
    sentence = re.sub(r' {2,}', ' ', sentence)                     # 중복 공백 제거 (원본 유지)
    sentence = sentence.strip()                                    # 양끝 공백 제거 (원본 유지)
    return sentence


questions = df["Q"].apply(preprocess_sentence).tolist()
answers = df["A"].apply(preprocess_sentence).tolist()

print(questions[0], "->", answers[0])
print(questions[1], "->", answers[1])

12시 땡! -> 하루가 또 가네요.
1지망 학교 떨어졌어 -> 위로해 드립니다.


### Step 2~3 (Decoder 기반 생성모델에 맞춘 챗봇 데이터 변형)

**여기가 "Decoder 기반의 생성모델임을 감안하여 챗봇 데이터를 변형" 하는 핵심 부분입니다.**

원본 노트북(Encoder-Decoder)에서는 질문(src)과 답변(tgt)을 **서로 다른 입력**으로 분리해
Encoder/Decoder에 각각 넣었습니다. 하지만 GPT-1은 Encoder가 없는 **단일 시퀀스 언어모델**이므로,
질문과 답변을 구분자(`<sep>`) 하나로 이어붙여 **하나의 문자열/시퀀스**로 합칩니다.

    <bos> 질문 토큰들 <sep> 답변 토큰들 <eos>

이렇게 만들면 모델은 "이 앞부분(질문+sep)이 주어졌을 때 다음 토큰(답변)이 무엇인지"를
예측하는 **순수 다음 토큰 예측(next-token prediction)** 문제로 챗봇 학습을 할 수 있습니다.

In [6]:
SEP_TOKEN = "<sep>"

# 토크나이저 학습용 코퍼스에는 문장을 그대로 사용 (sep는 특수기호로 별도 등록)
corpus_for_tokenizer = questions + answers


# 원본 generate_tokenizer()를 재사용하되, <sep> 토큰을 user_defined_symbols로 추가
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="chatbot",
                       pad_id=0,
                       bos_id=1,
                       eos_id=2,
                       unk_id=3,
                       user_defined_symbols=None):
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus:
            f.write(str(row) + '\n')

    train_args = (
        '--input=./%s --model_prefix=%s --vocab_size=%d '
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'
        % (file, model, vocab_size, pad_id, bos_id, eos_id, unk_id)
    )
    if user_defined_symbols:  # <sep> 같은 커스텀 특수 토큰 추가 (원본에 없던 변경점)
        train_args += ' --user_defined_symbols=%s' % ",".join(user_defined_symbols)

    spm.SentencePieceTrainer.Train(train_args)

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)
    return tokenizer


VOCAB_SIZE = 8000
tokenizer = generate_tokenizer(
    corpus_for_tokenizer,
    VOCAB_SIZE,
    lang="chatbot",
    user_defined_symbols=[SEP_TOKEN],
)

PAD_ID, BOS_ID, EOS_ID = 0, 1, 2
SEP_ID = tokenizer.piece_to_id(SEP_TOKEN)
print(f"vocab_size: {tokenizer.get_piece_size()}, sep_id: {SEP_ID}")


def build_sequence(q, a):
    """질문/답변을 <bos> Q <sep> A <eos> 형태의 단일 시퀀스로 결합"""
    q_ids = tokenizer.encode_as_ids(q)
    a_ids = tokenizer.encode_as_ids(a)
    return [BOS_ID] + q_ids + [SEP_ID] + a_ids + [EOS_ID]


sequences = [build_sequence(q, a) for q, a in zip(questions, answers)]

print("예시:", sequences[0])
print("복원:", tokenizer.decode_ids(sequences[0]))

vocab_size: 8000, sep_id: 4
예시: [1, 4288, 570, 6, 7824, 65, 4, 254, 10, 104, 84, 24, 5, 2]
복원: 12시 땡!<sep> 하루가 또 가네요.


### Step 5. 데이터 벡터화 (패딩 및 DataLoader 구성)

원본 `pad_sequences_custom()`을 그대로 재사용합니다.

In [7]:
MAX_LEN = 40

# 원본 pad_sequences_custom() 그대로 재사용
def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) > max_len:
            seq = seq[:max_len]
        else:
            seq = seq + [pad_value] * (max_len - len(seq))
        padded_sequences.append(seq)
    return torch.tensor(padded_sequences, dtype=torch.long)


seq_len_before_cut = [len(s) for s in sequences]
print(f"최대 시퀀스 길이: {max(seq_len_before_cut)}, "
      f"MAX_LEN({MAX_LEN}) 초과 비율: {sum(l > MAX_LEN for l in seq_len_before_cut) / len(seq_len_before_cut):.2%}")

padded_data = pad_sequences_custom(sequences, max_len=MAX_LEN, pad_value=PAD_ID)
print("padded_data shape:", padded_data.shape)

BATCH_SIZE = 64
train_dataset = TensorDataset(padded_data)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
print(f"배치 개수: {len(train_dataloader)}")

최대 시퀀스 길이: 44, MAX_LEN(40) 초과 비율: 0.03%
padded_data shape: torch.Size([11823, 40])
배치 개수: 185


## [평가기준 3] 모델의 입력 블록을 GPT 논문에 기반하여 수정

원본의 `positional_encoding()`(sinusoidal, 고정값)을 GPT-1 논문 수식
h0 = U·We + Wp 에 맞춰 **학습되는 Position Embedding**으로 교체합니다.
데이터에 위치 정보(position id)를 추가하는 과정을 아래에서 구현합니다.

In [8]:
class GPT1Embedding(nn.Module):
    """원본 Transformer.embedding()을 대체.
    변경점: sinusoidal positional_encoding(buffer) -> 학습되는 nn.Embedding(Wp)"""
    def __init__(self, vocab_size, d_model, max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)     # We
        self.pos_emb = nn.Embedding(max_seq_len, d_model)      # Wp (learned, 원본 대비 변경)
        self.do = nn.Dropout(dropout)

    def forward(self, x):
        # x: [batch_size, seq_len]
        batch, seq_len = x.shape
        # 데이터에 위치 정보(position id)를 추가하는 과정
        position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch, seq_len)

        out = self.token_emb(x) * math.sqrt(self.d_model)  # 원본과 동일하게 스케일링 유지
        out = out + self.pos_emb(position_ids)              # h0 = U*We + Wp
        return self.do(out)


# 입력이 정상적으로 구성되었는지 확인
_test_emb = GPT1Embedding(VOCAB_SIZE, d_model=64, max_seq_len=MAX_LEN)
_test_out = _test_emb(padded_data[:2])
print("입력 블록 출력 shape (batch, seq_len, d_model):", _test_out.shape)

입력 블록 출력 shape (batch, seq_len, d_model): torch.Size([2, 40, 64])


## [평가기준 4] GPT 모델 구성 (노드의 transformer 코드를 수정하여 GPT1 모델 구성)

### 4-1. MultiHeadAttention — 변경 없이 원본 그대로 재사용

In [9]:
# 원본 MultiHeadAttention 클래스, 변경 없음 (self-attention 계산 로직은 GPT-1에서도 동일)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        QK = torch.matmul(Q, K.transpose(-1, -2))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)
        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, _ = x.size()
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        out, attention_weights = self.scaled_dot_product_attention(WQ_splits, WK_splits, WV_splits, mask)
        out = self.combine_heads(out)
        out = self.linear(out)
        return out, attention_weights


print("MultiHeadAttention 준비 완료 (원본과 동일)")

MultiHeadAttention 준비 완료 (원본과 동일)


### 4-2. PoswiseFeedForwardNet — 활성함수만 GPT-1 논문 기준으로 수정 (ReLU → GELU)

In [10]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()  # 변경: 원본 nn.ReLU() -> GPT-1 논문 기준 nn.GELU()

    def forward(self, x):
        out = self.act(self.fc1(x))
        out = self.fc2(out)
        return out


print("PoswiseFeedForwardNet 준비 완료 (ReLU -> GELU 수정)")

PoswiseFeedForwardNet 준비 완료 (ReLU -> GELU 수정)


### 4-3. 마스크 생성 — Encoder/Cross 관련 마스크 제거, causal self-mask만 유지

In [11]:
# 원본 generate_padding_mask(), generate_lookahead_mask() 그대로 재사용
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    return (seq == PAD_ID).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    return torch.triu(torch.ones(size, size), diagonal=1)


# 변경: 원본 generate_masks()에서 enc_mask, dec_enc_mask를 제거하고
# self-attention용 causal mask 1개만 생성하도록 축소
def generate_self_mask(x: torch.Tensor) -> torch.Tensor:
    """
    x: [batch_size, seq_len]
    반환: [batch_size, 1, seq_len, seq_len]의 causal self-attention mask
    """
    seq_len = x.shape[1]
    padding_mask = generate_padding_mask(x)                       # [batch, 1, 1, seq_len]
    lookahead_mask = generate_lookahead_mask(seq_len).unsqueeze(0).unsqueeze(1)  # [1,1,seq_len,seq_len]

    padding_mask = padding_mask.to(x.device)
    lookahead_mask = lookahead_mask.to(x.device)

    return torch.max(padding_mask, lookahead_mask)


print("generate_self_mask 준비 완료 (enc_mask/dec_enc_mask 제거)")

generate_self_mask 준비 완료 (enc_mask/dec_enc_mask 제거)


### 4-4. GPT1DecoderLayer — 원본 DecoderLayer에서 enc_dec_attn 서브레이어 제거

In [12]:
class GPT1DecoderLayer(nn.Module):
    """원본 DecoderLayer 기반, enc_dec_attn(Cross-Attention) 서브레이어를 제거하고
    Masked Self-Attention -> FFN 2개 서브레이어만 남긴 GPT-1 Decoder 블록"""
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(GPT1DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        # self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)  # 제거됨 (Encoder 없음)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        # self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)  # enc_dec_attn용 norm 제거됨
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)     # ffn용 norm (원본의 norm_3 역할)

        self.do = nn.Dropout(dropout)

    def forward(self, x, self_mask):
        # Masked Multi-Head Self-Attention (원본과 동일한 Pre-LN 구조 유지)
        residual = x
        out = self.norm_1(x)
        out, self_attn = self.self_attn(out, out, out, mask=self_mask)
        out = self.do(out)
        out = out + residual

        # (원본에 있던 Encoder-Decoder Multi-Head Attention 블록은 여기서 제거됨)

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, self_attn


print("GPT1DecoderLayer 준비 완료 (Cross-Attention 제거)")

GPT1DecoderLayer 준비 완료 (Cross-Attention 제거)


### 4-5. GPT1Decoder — 원본 Decoder에서 enc_out, dec_enc_mask 인자 제거

In [13]:
class GPT1Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(GPT1Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [GPT1DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, self_mask):
        out = x
        self_attns = []
        for i in range(self.n_layers):
            out, self_attn = self.dec_layers[i](out, self_mask)  # enc_out 인자 제거됨
            self_attns.append(self_attn)
        return out, self_attns


print("GPT1Decoder 준비 완료 (Encoder 출력 의존성 제거)")

GPT1Decoder 준비 완료 (Encoder 출력 의존성 제거)


### 4-6. GPT1Model — 원본 Transformer 클래스에서 Encoder 제거 + 입력 블록 교체

In [15]:
class GPT1Model(nn.Module):
    """원본 Transformer 클래스 기반, Encoder 전체 삭제 + GPT1Embedding으로 입력 블록 교체 + Decoder만 사용"""
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len, dropout=0.1):
        super(GPT1Model, self).__init__()
        self.d_model = d_model

        # self.enc_emb = nn.Embedding(...)  # 제거됨 (Encoder 없음)
        self.embedding = GPT1Embedding(vocab_size, d_model, max_seq_len, dropout)  # 학습되는 position emb 사용

        # self.encoder = Encoder(...)  # 제거됨
        self.decoder = GPT1Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        # 추가: Pre-LN 구조는 마지막 블록 출력이 정규화되지 않은 채로 누적되므로,
        # 출력층 직전에 최종 LayerNorm(ln_f)을 추가 (GPT 계열 모델의 표준 구성)
        self.ln_f = nn.LayerNorm(d_model)

        self.fc = nn.Linear(d_model, vocab_size)
        # 논문의 P(u) = softmax(h_n * We^T): 출력 projection과 토큰 임베딩 weight 공유
        self.fc.weight = self.embedding.token_emb.weight

        # 추가: 논문 4.1절 "a simple weight initialization of N(0, 0.02) was sufficient"
        # 를 그대로 적용 (기본 nn.Embedding/nn.Linear 초기화를 사용하면 출력 logit의
        # 분산이 지나치게 커져 초기 loss가 비정상적으로 폭주하는 문제가 있었음)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, x, self_mask):
        """
        x: [batch_size, seq_len] (<bos> Q <sep> A <eos> 형태의 단일 시퀀스)
        self_mask: [batch_size, 1, seq_len, seq_len]
        """
        x_emb = self.embedding(x)
        dec_out, self_attns = self.decoder(x_emb, self_mask)
        dec_out = self.ln_f(dec_out)
        logits = self.fc(dec_out)
        return logits, self_attns


# 원본 대비 축소한 하이퍼파라미터 (Colab 무료 GPU/CPU에서도 학습 가능하도록 조정)
D_MODEL = 256    # (원 논문: 768). Colab GPU라면 256~512로 올려도 좋습니다.
N_HEADS = 4      # 원 논문: 12
D_FF = 1024       # (원 논문: 3072)
N_LAYERS = 4     # (원 논문: 12). Colab GPU라면 4~6으로 올려도 좋습니다.
DROPOUT = 0.2

model = GPT1Model(
    vocab_size=tokenizer.get_piece_size(),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    max_seq_len=MAX_LEN,
    dropout=DROPOUT,
).to(device)

# 모델이 정상적으로 구성되었는지 구조 출력
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {n_params:,}")

GPT1Model(
  (embedding): GPT1Embedding(
    (token_emb): Embedding(8000, 256)
    (pos_emb): Embedding(40, 256)
    (do): Dropout(p=0.2, inplace=False)
  )
  (decoder): GPT1Decoder(
    (dec_layers): ModuleList(
      (0-3): 4 x GPT1DecoderLayer(
        (self_attn): MultiHeadAttention(
          (W_q): Linear(in_features=256, out_features=256, bias=True)
          (W_k): Linear(in_features=256, out_features=256, bias=True)
          (W_v): Linear(in_features=256, out_features=256, bias=True)
          (linear): Linear(in_features=256, out_features=256, bias=True)
        )
        (ffn): PoswiseFeedForwardNet(
          (fc1): Linear(in_features=256, out_features=1024, bias=True)
          (fc2): Linear(in_features=1024, out_features=256, bias=True)
          (act): GELU(approximate='none')
        )
        (norm_1): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (norm_2): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (do): Dropout(p=0.2, inplace=False

### 4-7. 학습 설정 — LearningRateScheduler, loss_function은 원본 그대로 재사용

In [19]:
# 원본 LearningRateScheduler 그대로 재사용
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=200):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)


learning_rate = LearningRateScheduler(D_MODEL)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate(1), betas=(0.9, 0.98), eps=1e-9)


# 원본 loss_function 그대로 재사용
def loss_function(real, pred):
    real = real.to(device)
    pred = pred.to(device)

    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    loss_ = loss_.view(real.size())

    mask = (real != PAD_ID).float()
    loss_ = loss_ * mask
    return loss_.sum() / mask.sum()


# 변경: 원본 train_step(src, tgt, ...)에서 encoder 관련 인자를 모두 제거하고
# 단일 시퀀스(seq)에 대한 next-token 예측만 수행하도록 축소
def train_step(seq, model, optimizer, step_count):
    model.train()
    optimizer.zero_grad()

    seq = seq.to(device)
    dec_in = seq[:, :-1]   # 입력: 마지막 토큰 제외
    gold = seq[:, 1:]      # 정답: 한 칸 shift

    self_mask = generate_self_mask(dec_in)

    logits, self_attns = model(dec_in, self_mask)
    loss = loss_function(gold, logits)

    loss.backward()

    # 원본과 동일하게 step마다 learning rate 스케줄 적용
    for g in optimizer.param_groups:
        g['lr'] = learning_rate(step_count)
    optimizer.step()

    return loss


print("train_step 준비 완료 (Encoder/Cross-Attention 인자 제거)")

train_step 준비 완료 (Encoder/Cross-Attention 인자 제거)


In [23]:
EPOCHS = 15  # 검증용 값. Colab GPU에서는 10~20 epoch까지 늘리면 답변 품질이 더 좋아집니다.
step_count = 0

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)
    tqdm_bar = tqdm(total=dataset_count)

    for (seq,) in train_dataloader:
        step_count += 1
        loss = train_step(seq, model, optimizer, step_count)

        total_loss += loss.item()
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch + 1}, Loss: {total_loss / dataset_count:.4f}")

  0%|          | 0/185 [00:00<?, ?it/s]

Epoch 1, Loss: 1.3991


  0%|          | 0/185 [00:00<?, ?it/s]

Epoch 2, Loss: 2.2083


  0%|          | 0/185 [00:00<?, ?it/s]

...

Epoch 13, Loss: 1.0719


  0%|          | 0/185 [00:00<?, ?it/s]

Epoch 14, Loss: 1.0546


  0%|          | 0/185 [00:00<?, ?it/s]

Epoch 15, Loss: 1.0364


## [평가기준 5] 입력에 따른 출력 생성 확인

`<bos> 질문 <sep>` 까지만 넣어주고, 그 뒤를 모델이 자기회귀적으로 이어서
생성하는지 확인합니다.

In [24]:
@torch.no_grad()
def generate(model, question, max_new_tokens=30):
    model.eval()
    question = preprocess_sentence(question)
    prompt_ids = [BOS_ID] + tokenizer.encode_as_ids(question) + [SEP_ID]
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        cond_ids = input_ids[:, -MAX_LEN:]
        self_mask = generate_self_mask(cond_ids)
        logits, _ = model(cond_ids, self_mask)
        next_id = torch.argmax(logits[0, -1, :]).item()

        input_ids = torch.cat([input_ids, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == EOS_ID:
            break

    generated_ids = input_ids[0].tolist()
    # <sep> 이후, <eos> 이전까지만 답변으로 추출
    if SEP_ID in generated_ids:
        answer_ids = generated_ids[generated_ids.index(SEP_ID) + 1:]
    else:
        answer_ids = generated_ids
    answer_ids = [i for i in answer_ids if i != EOS_ID]
    return tokenizer.decode_ids(answer_ids)


test_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야.",
]

for q in test_questions:
    print(f"질문: {q}")
    print(f"답변: {generate(model, q)}")
    print("-" * 50)

질문: 지루하다, 놀러가고 싶어.
답변: 마음에 따라 얼마든지 바뀔 수 있어요.
--------------------------------------------------
질문: 오늘 일찍 일어났더니 피곤하다.
답변: 그런 날이 있더라고요.
--------------------------------------------------
질문: 간만에 여자친구랑 데이트 하기로 했어.
답변: 떨리을 때 매일 볼 수 있어요.
--------------------------------------------------
질문: 집에 있는다는 소리야.
답변: 시원하게  오세요.
--------------------------------------------------


아래 결과에 비해 warmup 수정은 학습률이 제대로 오르도록 고친 것이지, 모델의 근본적인 한계를 없애주는 건 아닙니다. 

warmup_step = 1000, epoch = 10

![](./6.png)

## 회고  
(질문, 답변)을 서로 다른 텐서로 넣던 것에서, <bos> 질문 <sep> 답변 <eos> 형태의 단일 시퀀스로 합치는 것으로 바뀌었다.  기존 코드는 sinusoidal(고정) positional encoding을 썼다. 위치 정보가 학습되지 않고 수식으로 미리 계산되어 있었다.  
GPT-1 논문은 이를 학습되는 Position Embedding으로 대체한다(h0 = U·We + Wp). 위치라는 개념 자체도 모델이 데이터로부터 배우게 만든 것이다.  
warmup_steps=1000을 원본 번역기 노트북에서 그대로 가져왔다가, 정작 챗봇 데이터(11,823개, epoch당 약 185 step)에는 전혀 맞지 않는 값이라는 걸 뒤늦게 알아챘다. 원본 노트북은 훨씬 큰 번역 데이터셋을 다뤘기 때문에 1000 step이 상대적으로 짧은 구간이었지만, 우리 데이터에서는 학습의 절반 이상을 "아직 학습률이 덜 오른 상태"로 흘려보내는 셈이었다. 하이퍼파라미터는 코드가 아니라 데이터 규모에 종속적인 값이라는 걸 다시 확인했다.  
warmup을 고쳤는데도 결과가 크게 안 바뀐 경험
설정 실수를 바로잡았다고 해서 결과물이 극적으로 좋아지는 건 아니었다. 실제로 warmup_steps를 200으로 낮추고 epoch을 15로 늘려 다시 돌려봤지만, 육안으로는 이전과 비슷하거나 오히려 문법이 어색한 답변도 하나 나왔다. 이 경험을 통해 "설정을 고치는 것"과 "성능이 실제로 개선되는 것"은 별개의 문제라는 걸 배웠다. 모델/데이터 규모의 한계, greedy decoding의 한계 같은 더 근본적인 요인이 있으면, 지엽적인 하이퍼파라미터 하나를 고친다고 결과가 확 달라지지 않는다.          
  
  아쉬운 점 / 다음에 시도해보고 싶은 것    
시간 관계상 이번엔 여기까지만 진행했지만, 다음에 이어서 해본다면 이런 것들을 시도해보고 싶다.
Validation split과 early stopping: 지금은 train loss만 보고 있어서, 모델이 일반화되고 있는지 그냥 외우고 있는지 구분이 안 된다.
Top-k/nucleus sampling 디코딩: 지금 greedy decoding은 안전하지만 "그런 날이 있더라고요"처럼 무난하고 뻔한 답으로 자꾸 회귀하는 경향이 있었다. 약간의 무작위성을 주면 이 문제가 완화될 것 같다.
Lexical Substitution Augmentation: 원본 노트북 Step 4에서 요구했던 augmentation을 이번엔 시간 관계상 생략했는데, 데이터가 11,823개로 크지 않은 만큼 데이터를 늘리는 게 과적합 완화에 도움이 될 것 같다.